# Demo: PyApprox integration with SPAROW

## OPF Models

This demonstrates how to use the sample allocation scheme recommended by PyApprox in the UQ workflow of SPAROW. 

This helps allocate finite computational budget between high-fidelity model evaluations and low-fidelity model evaluations, such that we achieve maximum variance reduction for the optimality gap estimator.

This tutorial was adapted from the PyApprox docs: https://sandialabs.github.io/pyapprox/multifidelity_estimation_cookbook.html 

In [1]:
import numpy as np

from pyapprox.statest.statistics import MultiOutputMean
from pyapprox.statest.mc_estimator import MCEstimator
from pyapprox.statest import MFMCEstimator
from pyapprox.statest.acv import default_allocator_factory
from pyapprox.statest.acv.base import FittedACVEstimator
from pyapprox.statest.allocation import MCAllocator
from pyapprox.optimization.minimize.scipy.slsqp import ScipySLSQPOptimizer

from sparow.conf_intervals.options import UQOptions
from sparow.conf_intervals.acv_mrp import ACVMRP
from sparow.conf_intervals.evaluate_true_optimality_gap import TrueOptimalityGapEvaluator
from sparow.conf_intervals.pyapprox_interface import (
    convert_pyapprox_allocation_to_acvmrp_params,
    build_pyapprox_mf_problem_from_ensemble,
)

from uq_opf import get_model_ensemble_for_uq

[    0.00] Initializing mpi-sppy
Alternative solutions package from or_topas is available.


In [ ]:
# ------------------------------------------------------
# User settings
# ------------------------------------------------------

MODEL_NAME = "HF"          # ignored by the ensemble builder; kept for interface consistency
LF_MODEL_TYPE = "dcopf"    # alternatives: "copperplate"

# One PyApprox sample = one scenario batch = one replication
BATCH_SIZE = 4
SOLVER_NAME = "ipopt"
SEED = 55

# Size of random HF batch used to generate the candidate xhat
XHAT_BATCH_SIZE = 1
XHAT_REPLICATION_ID = 999

# Pilot phase
N_PILOT = 10

# Total wall-clock budget used by PyApprox's allocation routine
TOTAL_BUDGET = 300.0

# Optional artificial delays to make cost differences easier to see in a demo
HF_COST_DELAY_SECONDS = 0.0
LF_COST_DELAY_SECONDS = 0.0

In [3]:
# ------------------------------------------------------
# Step 1: Build the HF/LF OPF ensemble
# ------------------------------------------------------
ensemble = get_model_ensemble_for_uq(
    model_name=MODEL_NAME,
    seed=SEED,
    with_replacement=True,
    lf_model_type=LF_MODEL_TYPE,
)

hf_model = ensemble.high_fidelity_model()
lf_model = ensemble.low_fidelity_model()

print("Built OPF multifidelity ensemble.")
print(f"HF model name: {hf_model.name()}, fidelity: {hf_model.fidelity()}")
print(f"LF model name: {lf_model.name()}, fidelity: {lf_model.fidelity()}")
print(f"Number of scenarios in full population: {len(hf_model.scenario_population().scenarios())}")

Built OPF multifidelity ensemble.
HF model name: HF, fidelity: high
LF model name: LF, fidelity: low
Number of scenarios in full population: 100


In [4]:
# ------------------------------------------------------
# Step 2: Generate candidate xhat from a small random HF SAA
# ------------------------------------------------------
# We intentionally do not use the full population here; instead we solve
# one HF SAA on a small random batch to obtain a realistic candidate.
xhat_scenarios = hf_model.draw_batch_of_scenarios(
    n=XHAT_BATCH_SIZE,
    replication_id=XHAT_REPLICATION_ID,
)

solved_hf = hf_model.solve_saa(
    sampled_scenarios=xhat_scenarios,
    solver_name=SOLVER_NAME,
    solver_options=None,
)

xhat = hf_model.get_first_stage_solution(solved_hf)

print("\nCandidate first-stage solution xhat extracted from one HF SAA on a random subset:")
print(f"Number of scenarios used to generate xhat: {XHAT_BATCH_SIZE}")
for k, v in xhat.items():
    print(f"  {k}: {v}")

INFO - 
INFO - ----------------------------------------------------------------------
INFO - ExtensiveFormSolver - START
INFO - Using single_bundle scheme (extensive form solve).
INFO - 
INFO - ----------------------------------------------------------------------
INFO - ExtensiveFormSolver - STOP



Candidate first-stage solution xhat extracted from one HF SAA on a random subset:
Number of scenarios used to generate xhat: 1
  time_periods[1].m.pg['1']: 2.00314050993904
  time_periods[1].m.pg['2']: 0.6524288332244026
  time_periods[1].m.pg['3']: 0.0
  time_periods[1].m.pg['4']: 0.0
  time_periods[1].m.pg['5']: 0.0
  time_periods[1].m.pg['6']: 0.0


In [5]:
# ------------------------------------------------------
# Step 3: Build PyApprox multifidelity problem
# ------------------------------------------------------
# SLSQP is often more robust than trust-constr for ACV allocation.
optimizer = ScipySLSQPOptimizer(maxiter=200)
allocator_factory = lambda est: default_allocator_factory(est, optimizer=optimizer)

problem, bkd = build_pyapprox_mf_problem_from_ensemble(
    ensemble=ensemble,
    xhat=xhat,
    batch_size=BATCH_SIZE,
    solver_name=SOLVER_NAME,
    solver_options=None,
    seed=SEED,
    hf_cost_delay_seconds=HF_COST_DELAY_SECONDS,
    lf_cost_delay_seconds=LF_COST_DELAY_SECONDS,
)

models = problem.models()
variable = problem.prior()
costs = problem.costs()
nmodels = len(models)
nqoi = models[0].nqoi()

print("\nModel wrapper types:")
for idx, model in enumerate(models):
    print(idx, type(model), callable(model))

# These are the estimated average wall-clock costs per replication-level evaluation.
costs_np = bkd.to_numpy(costs)
print("\nEstimated model costs:")
for a, c in enumerate(costs_np):
    print(f"  model {a}: estimated cost = {c:.6f}")

INFO - 
INFO - ----------------------------------------------------------------------
INFO - ExtensiveFormSolver - START
INFO - Using single_bundle scheme (extensive form solve).
INFO - 
INFO - ----------------------------------------------------------------------
INFO - ExtensiveFormSolver - STOP
INFO - 
INFO - ----------------------------------------------------------------------
INFO - ExtensiveFormSolver - START
INFO - Using single_bundle scheme (extensive form solve).
INFO - 
INFO - ----------------------------------------------------------------------
INFO - ExtensiveFormSolver - STOP
INFO - 
INFO - ----------------------------------------------------------------------
INFO - ExtensiveFormSolver - START
INFO - Using single_bundle scheme (extensive form solve).
INFO - 
INFO - ----------------------------------------------------------------------
INFO - ExtensiveFormSolver - STOP
INFO - 
INFO - ----------------------------------------------------------------------
INFO - ExtensiveF


Model wrapper types:
0 <class 'sparow.conf_intervals.pyapprox_interface.PyApproxModelWrapper'> True
1 <class 'sparow.conf_intervals.pyapprox_interface.PyApproxModelWrapper'> True

Estimated model costs:
  model 0: estimated cost = 1.431227
  model 1: estimated cost = 0.603361


In [6]:
# ------------------------------------------------------
# Step 4: Pilot evaluation for covariance estimation
# ------------------------------------------------------
np.random.seed(42)

# Draw pilot scenario batches; each column is one batch / replication.
samples_pilot = variable.rvs(N_PILOT)

# Evaluate every model on the same pilot batches.
vals_pilot = [m(samples_pilot) for m in models]

stat = MultiOutputMean(nqoi, bkd)
cov_pilot, = stat.compute_pilot_quantities(vals_pilot)
stat.set_pilot_quantities(cov_pilot)

cov_np = bkd.to_numpy(cov_pilot)
print("\nPilot covariance matrix:")
print(cov_np)

print("\nPilot correlations with HF model:")
for a in range(1, nmodels):
    rho = cov_np[0, a] / np.sqrt(cov_np[0, 0] * cov_np[a, a])
    print(f"  Pilot correlation ρ(f0, f{a}) = {rho:.4f}")

INFO - 
INFO - ----------------------------------------------------------------------
INFO - ExtensiveFormSolver - START
INFO - Using single_bundle scheme (extensive form solve).
INFO - 
INFO - ----------------------------------------------------------------------
INFO - ExtensiveFormSolver - STOP
INFO - 
INFO - ----------------------------------------------------------------------
INFO - ExtensiveFormSolver - START
INFO - Using single_bundle scheme (extensive form solve).
INFO - 
INFO - ----------------------------------------------------------------------
INFO - ExtensiveFormSolver - STOP
INFO - 
INFO - ----------------------------------------------------------------------
INFO - ExtensiveFormSolver - START
INFO - Using single_bundle scheme (extensive form solve).
INFO - 
INFO - ----------------------------------------------------------------------
INFO - ExtensiveFormSolver - STOP
INFO - 
INFO - ----------------------------------------------------------------------
INFO - ExtensiveF


Pilot covariance matrix:
[[34592379.70328547 22924955.4076236 ]
 [22924955.4076236  16878422.18957545]]

Pilot correlations with HF model:
  Pilot correlation ρ(f0, f1) = 0.9488


In [7]:
# ------------------------------------------------------
# Step 5: PyApprox allocation
# ------------------------------------------------------
pilot_cost = float(costs_np.sum()) * N_PILOT
remaining = TOTAL_BUDGET - pilot_cost

print("\nBudget summary:")
print(f"Pilot cost: {pilot_cost}")
print(f"Remaining budget: {remaining}")

est = MFMCEstimator(stat, costs)
allocator = allocator_factory(est)
result = allocator.allocate(remaining)
fitted = FittedACVEstimator(est, result)

print(f"\nPyApprox samples per model (HF total, LF total): {fitted.nsamples_per_model()}")

m, M = convert_pyapprox_allocation_to_acvmrp_params(fitted.nsamples_per_model())
print("Translated ACV-MRP counts:")
print(f"  Number of paired replications: {m}")
print(f"  Number of additional LF replications: {M}")

print(f"Predicted PyApprox std: {float(fitted.covariance()[0, 0])**0.5:.6f}")


Budget summary:
Pilot cost: 20.345881462097168
Remaining budget: 279.65411853790283

PyApprox samples per model (HF total, LF total): [ 66 306]
Translated ACV-MRP counts:
  Number of paired replications: 66
  Number of additional LF replications: 240
Predicted PyApprox std: 392.558269


In [8]:
# ------------------------------------------------------
# Step 6: Evaluate PyApprox estimator on allocated samples
# ------------------------------------------------------
samples_per_model = fitted.generate_samples_per_model(variable.rvs)
print(f"\nAllocated sample shapes: {[s.shape for s in samples_per_model]}")

values_per_model = [models[a](samples_per_model[a]) for a in range(nmodels)]

estimate = fitted(values_per_model)
estimate_scalar = np.asarray(estimate).item()

print("\nPyApprox point estimator:")
print(f"Estimated mean of HF replication outputs, E[F_n(xhat)]: {estimate_scalar:.6f}")

INFO - 
INFO - ----------------------------------------------------------------------
INFO - ExtensiveFormSolver - START
INFO - Using single_bundle scheme (extensive form solve).



Allocated sample shapes: [(4, 66), (4, 306)]


INFO - 
INFO - ----------------------------------------------------------------------
INFO - ExtensiveFormSolver - STOP
INFO - 
INFO - ----------------------------------------------------------------------
INFO - ExtensiveFormSolver - START
INFO - Using single_bundle scheme (extensive form solve).
INFO - 
INFO - ----------------------------------------------------------------------
INFO - ExtensiveFormSolver - STOP
INFO - 
INFO - ----------------------------------------------------------------------
INFO - ExtensiveFormSolver - START
INFO - Using single_bundle scheme (extensive form solve).
INFO - 
INFO - ----------------------------------------------------------------------
INFO - ExtensiveFormSolver - STOP
INFO - 
INFO - ----------------------------------------------------------------------
INFO - ExtensiveFormSolver - START
INFO - Using single_bundle scheme (extensive form solve).
INFO - 
INFO - ----------------------------------------------------------------------
INFO - ExtensiveF

KeyboardInterrupt: 

In [ ]:
# ------------------------------------------------------
# Step 7: Run ACV-MRP with translated (m, M)
# ------------------------------------------------------
options = UQOptions(
    n=BATCH_SIZE,
    m=m,
    M=M,
    alpha=0.05,
    seed=SEED,
    with_replacement=True,
    solver_name=SOLVER_NAME,
    verbose=True,
)

acv = ACVMRP(
    hf_model=ensemble.high_fidelity_model(),
    lf_model=ensemble.low_fidelity_model(),
    options=options,
)

results = acv.run(xhat=xhat)

print("\nACV-MRP results:")
print(f"ACV-MRP Point estimate: {results['point_estimate']}")
print(f"HF-only point estimate: {results['point_estimate_hf_only']}")
print(f"CI: [{results['ci_lower']}, {results['ci_upper']}]")
print(f"Estimated control variate coefficient: {results['control_variate_coefficient']}")
print(f"Estimated sample correlation: {results['sample_correlation']}")
print(f"Variance reduction factor: {results['variance_reduction_factor']}")

In [ ]:
# ------------------------------------------------------
# Step 8: True finite-population HF gap
# ------------------------------------------------------
true_gap_evaluator = TrueOptimalityGapEvaluator(
    model=ensemble.high_fidelity_model(),
    solver_name=SOLVER_NAME,
    solver_options=None,
)

true_gap_results = true_gap_evaluator.compute_true_gap(xhat=xhat)

print("\nTrue finite-population HF quantities:")
print(f"True optimal value: {true_gap_results['true_optimal_value']}")
print(f"xhat true value: {true_gap_results['xhat_true_value']}")
print(f"True optimality gap: {true_gap_results['true_gap']}")

In [ ]:
# ------------------------------------------------------
# Step 9: Compare MF prediction against HF-only MC under same budget
# ------------------------------------------------------
stat_mc = MultiOutputMean(nqoi, bkd)
stat_mc.set_pilot_quantities(cov_pilot[:1, :1])

# HF-only estimator uses only model 0 cost.
mc_est = MCEstimator(stat_mc, costs[:1])
mc_fitted = MCAllocator(mc_est).allocate(remaining)

mc_var = float(mc_fitted.covariance()[0, 0])
mf_var = float(fitted.covariance()[0, 0])

print("\nHF-only MC vs PyApprox MF under same budget:")
print(f"HF-only MC std: {mc_var**0.5}")
print(f"PyApprox MF std: {mf_var**0.5}")
print(f"Variance reduction: {mc_var / mf_var:.2f}×")